# 🤖 Push-T Keypoint Alignment with World Model Predictive Control (MPC)

This notebook implements a **Keypoint-based Model Predictive Control (MPC)** planner in state-space to push a T-shaped block onto a target T-shaped outline, aligning specific control keypoints.

### Interactive & Simulation Goals:
1. **Grey Outline Target**: The target T is styled as a grey outline.
2. **Royal Blue Alphabet Block**: The movable T-block is styled in deep royal blue.
3. **Two Keypoint Dots**: We attach two control keypoints onto the movable T-block (Red and Blue dots) and two corresponding target keypoints on the grey target T-outline.
4. **Real-time Predictive World Model**: We implement a lightweight physics model and a **Vector-Field MPC** predictive planner. The agent pusher (orange circle) plans candidate actions to push the block, minimizing the distance between the keypoints!
5. **OpenCV Overlays**: We overlay the glowing keypoint dots and the alignment error lines directly on top of the rendered simulation frames for stunning visual feedback!

In [ ]:
import os
import warnings
import numpy as np
import cv2
import pygame
import matplotlib
# Set Agg backend for headless compatibility
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import gymnasium as gym
from stable_worldmodel.envs.pusht.env import PushT

# Ensure warnings are ignored
warnings.filterwarnings("ignore")

In [ ]:
# 1. Initialize the Push-T environment in headless mode
env = PushT(render_mode='rgb_array')

# 2. Customize the visual layouts
# Goal Layout 'T' color -> Grey (neutral [128, 128, 128])
env.variation_space['goal']['color'].set_init_value(np.array([128, 128, 128], dtype=np.uint8))

# Background canvas color -> White [255, 255, 255]
env.variation_space['background']['color'].set_init_value(np.array([255, 255, 255], dtype=np.uint8))

# Agent pusher color -> Orange [230, 90, 40]
env.variation_space['agent']['color'].set_init_value(np.array([230, 90, 40], dtype=np.uint8))

# Block 'T' color -> Deep Royal Blue [30, 144, 255]
env.variation_space['block']['color'].set_init_value(np.array([30, 144, 255], dtype=np.uint8))

print("✅ Environment customized successfully.")

In [ ]:
# 3. Implement Keypoint Math, Lightweight Physics Predictor, and Vector-Field Planner

kpA_off = np.array([-45.0, -45.0]) # Top-left of T crossbar
kpB_off = np.array([0.0, 75.0])    # Bottom of T stem

def get_keypoints(x, y, angle):
    """Compute global coordinates of the two keypoints on the T body."""
    cos, sin = np.cos(angle), np.sin(angle)
    ax = x + kpA_off[0] * cos - kpA_off[1] * sin
    ay = y + kpA_off[0] * sin + kpA_off[1] * cos
    bx = x + kpB_off[0] * cos - kpB_off[1] * sin
    by = y + kpB_off[0] * sin + kpB_off[1] * cos
    return np.array([ax, ay]), np.array([bx, by])

def step_physics(bx, by, b_angle, px, py, vx, vy):
    """Lightweight state-space physics predictor simulating pusher-block collision."""
    # Move pusher
    px_new = px + vx
    py_new = py + vy

    # Collision detection in local coordinates
    dx = px_new - bx
    dy = py_new - by
    lx = dx * np.cos(-b_angle) - dy * np.sin(-b_angle)
    ly = dx * np.sin(-b_angle) + dy * np.cos(-b_angle)

    # Distance to Box 1 (Crossbar): x in [-60, 60], y in [-60, -30]
    cx1 = np.clip(lx, -60, 60)
    cy1 = np.clip(ly, -60, -30)
    dist1 = np.hypot(lx - cx1, ly - cy1)

    # Distance to Box 2 (Stem): x in [-15, 15], y in [-30, 60]
    cx2 = np.clip(lx, -15, 15)
    cy2 = np.clip(ly, -30, 60)
    dist2 = np.hypot(lx - cx2, ly - cy2)

    if dist1 < dist2:
        cx, cy, minDist = cx1, cy1, dist1
    else:
        cx, cy, minDist = cx2, cy2, dist2
 
    radius = 18.0
    if minDist < radius:
        overlap = radius - minDist
        nlx = lx - cx
        nly = ly - cy
        length = np.hypot(nlx, nly)
        if length == 0:
            nlx, nly, length = 0.0, -1.0, 1.0
        nlx /= length
        nly /= length

        # Convert local normal to global normal
        cos_g = np.cos(b_angle)
        sin_g = np.sin(b_angle)
        gnx = nlx * cos_g - nly * sin_g
        gny = nlx * sin_g + nly * cos_g

        # Translate block
        bx -= gnx * overlap * 0.85
        by -= gny * overlap * 0.85
 
        # Calculate torque based on cross product
        gcx = cx * cos_g - cy * sin_g
        gcy = cx * sin_g + cy * cos_g
        forceX = -gnx * overlap
        forceY = -gny * overlap
        torque = gcx * forceY - gcy * forceX

        b_angle += torque * 0.00035

    return bx, by, b_angle, px_new, py_new

def plan_cem(block_state, pusher_state, target_pose, num_samples=30, horizon=6):
    """Vector field planner for highly robust and direct keypoint alignment."""
    bx, by, b_angle = block_state
    px, py = pusher_state
    tx, ty, tangle = target_pose

    dx = tx - bx
    dy = ty - by
    dist = np.hypot(dx, dy)

    # Target push direction
    ux = dx / (dist + 1e-5)
    uy = dy / (dist + 1e-5)
    if dist < 5.0:
        ux = np.cos(tangle)
        uy = np.sin(tangle)

    # Rotation error
    d_angle = tangle - b_angle
    d_angle = np.arctan2(np.sin(d_angle), np.cos(d_angle))

    # Perpendicular vector to push direction
    upx = -uy
    upy = ux

    # Contact offset for torque
    K_rot = 45.0
    offset = -np.clip(d_angle * K_rot, -35.0, 35.0)

    # Target contact point on block boundary
    cx = bx - ux * 55.0 + upx * offset
    cy = by - uy * 55.0 + upy * offset

    # Pusher to contact point
    gX = cx - px
    gY = cy - py
    dist_to_contact = np.hypot(gX, gY)

    # Pusher to block center
    uX = bx - px
    uY = by - py
    dist_to_block = np.hypot(uX, uY)

    if dist_to_contact > 20.0:
        # Navigation Mode
        gux = gX / (dist_to_contact + 1e-5)
        guy = gY / (dist_to_contact + 1e-5)

        if dist_to_block < 65.0:
            # Steering around the block
            n1x, n1y = -uY / dist_to_block, uX / dist_to_block
            n2x, n2y = uY / dist_to_block, -uX / dist_to_block
            dot1 = n1x * gux + n1y * guy
            
            nx = n1x if dot1 >= 0 else n2x
            ny = n1y if dot1 >= 0 else n2y

            w = (65.0 - dist_to_block) / 65.0
            vx = (1.0 - w) * gux + w * nx
            vy = (1.0 - w) * guy + w * ny
        else:
            vx = gux
            vy = guy

        speed = 5.0
        v_len = np.hypot(vx, vy)
        vx = (vx / (v_len + 1e-5)) * speed
        vy = (vy / (v_len + 1e-5)) * speed
    else:
        # Pushing Mode
        vx = ux * 3.5 + gX * 0.15
        vy = uy * 3.5 + gY * 0.15

        speed = np.hypot(vx, vy)
        if speed > 4.5:
            vx = (vx / speed) * 4.5
            vy = (vy / speed) * 4.5

    return vx, vy

print("✅ Vector-Field MPC Keypoint Planner loaded successfully.")

In [ ]:
# 4. Run closed-loop keypoint alignment simulation and capture frames with OpenCV overlays

obs, info = env.reset(seed=42)

# Extract block Random starting state and Goal pose
state = obs['state']
block_state = state[2:5] # [bx, by, b_angle]
goal_state = info['goal_state']
goal_pose = info['goal_pose'] # [tx, ty, tangle]

# Spawn pusher end-effector close to block to guarantee quick interaction
pushVec = goal_pose[:2] - block_state[:2]
pushVec /= np.linalg.norm(pushVec) + 1e-5
agent_pos = block_state[:2] - pushVec * 60

new_state = np.concatenate([agent_pos, block_state[:2], [block_state[2]], [0.0, 0.0]])
env._set_state(new_state)

# Setup simulation
frames = []
max_steps = 100

print("Running Vector Field MPC Keypoint alignment simulation...")
for step in range(max_steps):
    # Get current state from environment
    obs_info = env._get_info()
    bx, by, b_angle = obs_info['block_pose']
    px, py = obs_info['pos_agent']
    tx, ty, tangle = obs_info['goal_pose']
    
    # 1. Run Vector Field planner
    best_vx, best_vy = plan_cem(
        (bx, by, b_angle), (px, py), (tx, ty, tangle)
    )
    
    # 2. Step the gymnasium environment using relative action
    # PushT actions are continuous velocity vectors scaled by action_scale=100
    action = np.array([best_vx, best_vy]) / 100.0
    obs, reward, terminated, truncated, info = env.step(action)

    # 3. Capture frame and draw OpenCV Keypoint overlays
    raw_frame = env.render() # (224, 224, 3) uint8 numpy
    frame = raw_frame.copy()
    
    # Scale keypoint coordinates from 512x512 physical window to 224x224 render canvas
    scale = 224.0 / 512.0
    kps = get_keypoints(bx, by, b_angle)
    targets = get_keypoints(tx, ty, tangle)
    
    kpA = (kps[0] * scale).astype(np.int32)
    kpB = (kps[1] * scale).astype(np.int32)
    tA = (targets[0] * scale).astype(np.int32)
    tB = (targets[1] * scale).astype(np.int32)
    
    # Draw Keypoint targets (Grey dots)
    cv2.circle(frame, tuple(tA), 4, (120, 120, 120), -1)
    cv2.circle(frame, tuple(tB), 4, (120, 120, 120), -1)
    
    # Draw alignment error lines (Dashed/thin lines)
    cv2.line(frame, tuple(kpA), tuple(tA), (40, 90, 230), 1)
    cv2.line(frame, tuple(kpB), tuple(tB), (40, 90, 230), 1)
    
    # Draw glowing Red (Keypoint A) and Blue (Keypoint B) dots
    cv2.circle(frame, tuple(kpA), 5, (0, 0, 255), -1)
    cv2.circle(frame, tuple(kpB), 5, (255, 0, 0), -1)
    
    frames.append(frame)
    
    # Calculate MSE
    errA = np.linalg.norm(kps[0] - targets[0])
    errB = np.linalg.norm(kps[1] - targets[1])
    mse = (errA**2 + errB**2) / 2.0
    
    if step % 10 == 0 or mse < 35.0:
        print(f"Step {step:02d}: Keypoint Alignment MSE = {mse:.2f}")
        
    if mse < 35.0:
        print(f"🎉 Goal reached successfully in {step} steps! Keypoint Alignment MSE = {mse:.2f}")
        break
    
print(f"Simulation complete. Captured {len(frames)} overlay frames.")

In [ ]:
# 5. Render and display the keypoint alignment animation directly in Jupyter
fig, ax = plt.subplots(figsize=(6, 6))
ax.axis('off')

# Initial plot
im = ax.imshow(frames[0])

def update(frame_idx):
    im.set_array(frames[frame_idx])
    return [im]

ani = animation.FuncAnimation(
    fig, update, frames=len(frames), interval=100, blit=True
)

# Close static plot to prevent duplicate displays
plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())